# Byte Model Evaluation: What Would Count as a Win?

Allow 60-90 minutes. Prerequisites: the conversion lab, negative log-likelihood,
scaling-law experiments, and basic latency measurements. The exercise builds
four small tools that expose mismatched comparisons. The notebook then uses
them on explicitly synthetic observations, not claimed paper results.

## 1. Choose a common denominator

A model predicts units, but the user consumes text. Different tokenizers give
different counts for the same text. Normalize the **sum** of negative
log-likelihoods by raw UTF-8 content bytes:

```text
BPB = sum(NLL in nats) / (content_bytes * ln(2))
```

Four predictions each costing 2 bits and eight predictions each costing 1 bit
both total 8 bits. If the same content is 4 bytes, both have 2 BPB. Averaging
per prediction first would give an incompatible denominator.

For byte-plus-EOT models, include losses for predicted EOT symbols, but do not
count EOT, BOS, or padding as content bytes. This scores the declared augmented
sequence. It is not the raw-text likelihood marginalized over every possible
segmentation. Report the convention instead of silently switching representations.

`bits_per_byte` takes nonnegative finite NLL values and a positive integer byte
count. Empty inputs or fractional counts should fail, not produce a reassuring
number. Document which symbols contributed to the numerator and denominator.

## 2. A likelihood metric is not a quality verdict

Imagine the correct answer has probability 0.4 in two models. In model A, every
other answer has less than 0.4; in model B, one wrong answer has 0.5. The target
loss is identical, but greedy predictions disagree. Task accuracy, output
validity, and critical error slices need direct evaluation.

The tiny training notebook reports representation BPB. It cannot establish
general language ability, reliable reasoning, multilingual quality, or safety.
Use an independently reviewed task set before making a product decision.

## 3. Name an observation, interpolation, or extrapolation

Suppose you trained at compute budgets 10, 20, and 40. A quality result actually
evaluated at 20 is observed. An estimate at 30 is interpolated. An estimate at
100 is extrapolated. A fitted curve may be useful, but it does not create more
trained checkpoints. `evidence_region` labels the compute location only; even a
curve prediction at a sampled budget is still distinct from its measured score.

Before accepting a predicted crossover, inspect the largest measured budget,
fit residuals, held-out fit points, seed variation, and sensitivity to the curve
family and validation set. A fitted asymptote is not a universal upper bound.
Do not treat a ceiling predicted for one experimental setup as a law of bytes.

In the accompanying notebook, leave out the largest observation, fit a simple
curve using only the earlier points, then evaluate the held-out prediction.
Change a single noisy observation and inspect the extrapolated result. This is
a sensitivity demonstration, not a validated scaling law.

## 4. Compute logit payload honestly

Our storage exercise isolates target-logit payloads:

```text
dense_FP32 = positions * vocabulary_size * 4
sparse_top_k = positions * k * (4-byte value + 4-byte index)
```

At 10 prediction positions, a 1,000-entry dense vocabulary costs 40,000 bytes.
Keeping five logits with indices costs 400 bytes. But a byte representation of
the same content may need 45 positions. Its 256-entry dense target then costs
46,080 bytes. A small vocabulary alone is not a guaranteed storage win against
aggressive top-k truncation. Compare both storage and the probability mass lost.

The exercise excludes token IDs, text storage, compression, metadata, and
teacher inference. Real cost accounting must add them. Its numbers are original
teaching examples, not the paper's reported savings.

## 5. Measure equal completed work

`serving_summary` calculates sequential aggregate content throughput:

```text
bytes_per_second = 1000 * sum(completed_content_bytes) / sum(elapsed_ms)
```

Do not average individual request rates. A 10-byte request taking 100 ms and a
90-byte request taking 300 ms complete 100 bytes in 400 ms: 250 bytes/s, not the
unweighted average of their separate rates. For p95, this lab explicitly uses
the nearest-rank sample at `ceil(0.95*n)-1`; a production dashboard may use a
different percentile estimator.

The training notebook times two warmups and five sequential CPU requests for
eight returned ASCII bytes each. It constrains byte outputs to the synthetic
alphabet (plus EOT where applicable), uses no KV cache, and reports incomplete
requests instead of pretending they completed. The generation cap is a safety
bound, not a meaningful production SLO. Five samples are only a smoke test.
Parameter tensor storage excludes activations, KV cache, runtime overhead, and
peak RAM; do not label it process memory.

For a real deployment experiment, record model/checkpoint and tokenizer,
hardware/runtime/precision, prompt length in content bytes, requested output
length, batch and concurrency policy, warmup, at least a justified number of
repetitions, time-to-first-byte, inter-byte latency, end-to-end p50/p95, peak
memory, failures, and quality by task slice. Include teacher-label generation,
student training, and recurring serving costs in a lifetime comparison.

## 6. Make a conditional decision

Write a short evidence memo after the notebook:

1. Which rows are measured, and which depend on a fitted curve?
2. Are content, compute, model capacity, and quality each matched or explicitly
   allowed to differ? Name the controlled axis for every comparison.
3. Which method meets the target quality floor and latency/memory constraints?
4. What result would reverse your recommendation?

The correct answer can be **insufficient evidence**. A method that conserves
teacher probabilities can still lose on end-to-end cost or task quality.

## Worked checks and AI-assisted practice

Hand-calculate the 8-bit/4-byte example and the 100-byte/400-ms example first.
Then implement the four functions and run their grader. The worked solution
follows the same contracts and does not relax invalid-input checks. Ask an AI
assistant to propose counterexamples, but verify them against the declared
units and formulas yourself. Do not accept invented benchmark observations.

**Exit check:** give one reason each that smaller vocabulary, lower BPB, and
higher prediction-units/second can fail to imply a better deployed model.

## Research context

[Breaking the Token Ceiling, sections 7-10 and appendices A/D/G](https://arxiv.org/abs/2609.12303)
is the case study, not a production recommendation. Several headline comparisons
use asymptotic extrapolation; inference-cost parity is not established by the
study. The course keeps those claims separate from both measured checkpoints
and its own synthetic experiments.


## Work first, then compare

CPU runtime is sufficient for the exercises. NumPy and PyTorch are available in Colab. Run the starter and grading cells after implementing the TODOs. An untouched starter is expected to fail. The reference solution below replaces the functions only when you deliberately run that cell. Then rerun the checks. No account, key, model download, or paid GPU is required.


In [ ]:
"""Compare raw content, not incompatible token counts. No model is needed."""
import numpy as np

def bits_per_byte(nll_nats, content_bytes):
    """Sum a nonempty 1-D vector of finite nonnegative NLLs / (bytes * ln(2)).
    content_bytes is a positive integer excluding BOS/EOT/padding symbols.
    Invalid inputs raise ValueError. NLL includes any predicted EOT symbols.
    This scores the declared representation, not marginalized raw-text likelihood.
    """
    raise NotImplementedError

def logit_storage(positions, vocab_size, top_k=None):
    """Logit payload bytes: dense FP32, or top-k FP32 values + int32 indices.
    positions and vocab_size must be positive integers. top_k is None or an
    integer 1..vocab_size. Invalid inputs raise ValueError. Excludes text,
    token IDs, filesystem metadata, compression, and teacher-generation cost.
    """
    raise NotImplementedError

def serving_summary(content_bytes, elapsed_ms):
    """Equal-length nonempty 1-D lists for completed SEQUENTIAL requests.
    Byte counts are positive integers; times finite and positive. Return dict:
    bytes_per_second = 1000 * total bytes / total milliseconds;
    p95_ms = sorted times[ceil(0.95 * n) - 1] (nearest-rank convention).
    Reject invalid inputs with ValueError. No concurrency claim is made.
    """
    raise NotImplementedError

def evidence_region(measured_flops, target_flops):
    """Positive finite compute values; measured_flops is a nonempty 1-D list.
    Return 'measured' for an exact sampled budget, 'interpolated' strictly
    inside its min/max, otherwise 'extrapolated'. Invalid inputs: ValueError.
    This labels the location, not the accuracy or confidence of a fitted model.
    """
    raise NotImplementedError


In [ ]:
import numpy as np
np.testing.assert_allclose(bits_per_byte([np.log(2), 3*np.log(2)], 2), 2)
assert bits_per_byte([0, 0], 3) == 0
# Same content, different number of prediction units: sum losses first.
np.testing.assert_allclose(bits_per_byte([np.log(2)]*8, 4), bits_per_byte([2*np.log(2)]*4, 4))
assert logit_storage(10, 1000) == 40000
assert logit_storage(10, 1000, 5) == 400
assert logit_storage(10, 1000, 1000) == 80000
assert logit_storage(45, 256) == 46080
_counts, _times = [10, 90], [100., 300.]
assert serving_summary(_counts, _times) == {'bytes_per_second': 250., 'p95_ms': 300.}
assert serving_summary([1]*20, list(range(1,21)))['p95_ms'] == 19
assert _counts == [10,90] and _times == [100.,300.]
for _budget, _region in [(10,'measured'),(20,'measured'),(15,'interpolated'),(9,'extrapolated'),(100,'extrapolated')]:
    assert evidence_region([20, 10, 20], _budget) == _region
assert evidence_region([10], 11) == 'extrapolated'
for _call in [
    lambda: bits_per_byte([], 1), lambda: bits_per_byte([-1], 1),
    lambda: bits_per_byte([np.nan], 1), lambda: bits_per_byte([1], 0),
    lambda: bits_per_byte([1], 1.5), lambda: bits_per_byte([1], True),
    lambda: logit_storage(2.5, 256), lambda: logit_storage(1, 0),
    lambda: logit_storage(1, 256, 257), lambda: logit_storage(1, 256, True),
    lambda: serving_summary([1], [0]), lambda: serving_summary([1.5], [1]),
    lambda: serving_summary([1], [1, 2]), lambda: serving_summary([], []),
    lambda: serving_summary([1], [np.inf]), lambda: evidence_region([], 1),
    lambda: evidence_region([0, 1], 1), lambda: evidence_region([1], np.nan),
]:
    try:
        _call()
    except ValueError:
        pass
    else:
        raise AssertionError('invalid measurement must raise ValueError')
print('BPB denominator, dense/sparse payload, sequential throughput, p95 and evidence regions passed.')


## Worked reference solution

Compare the state transitions and invariants with your implementation. Passing output tests alone does not establish memory efficiency or reproduce the paper's performance results.


In [ ]:
import numpy as np
from numbers import Integral

def _positive_integer(value):
    return isinstance(value, Integral) and not isinstance(value, (bool, np.bool_)) and value > 0

def bits_per_byte(nll_nats, content_bytes):
    losses = np.asarray(nll_nats, dtype=float)
    if (losses.ndim != 1 or not losses.size or not np.isfinite(losses).all()
            or (losses < 0).any() or not _positive_integer(content_bytes)):
        raise ValueError('expected NLL values and a positive content-byte count')
    return float(losses.sum() / (content_bytes * np.log(2)))

def logit_storage(positions, vocab_size, top_k=None):
    if not _positive_integer(positions) or not _positive_integer(vocab_size):
        raise ValueError('positive integer dimensions required')
    if top_k is None:
        return int(positions * vocab_size * 4)
    if not _positive_integer(top_k) or top_k > vocab_size:
        raise ValueError('top_k must be an integer within vocabulary size')
    return int(positions * top_k * 8)

def serving_summary(content_bytes, elapsed_ms):
    counts, times = np.asarray(content_bytes), np.asarray(elapsed_ms, dtype=float)
    if (counts.ndim != 1 or times.shape != counts.shape or not counts.size
            or counts.dtype.kind not in 'iu' or (counts <= 0).any()
            or not np.isfinite(times).all() or (times <= 0).any()):
        raise ValueError('expected positive byte counts and completed request times')
    return {'bytes_per_second': float(1000 * counts.sum() / times.sum()),
            'p95_ms': float(np.sort(times)[int(np.ceil(.95 * len(times))) - 1])}

def evidence_region(measured_flops, target_flops):
    values = np.asarray(measured_flops, dtype=float)
    if (values.ndim != 1 or not values.size or not np.isfinite(values).all()
            or (values <= 0).any() or not np.isscalar(target_flops)
            or not np.isfinite(target_flops) or target_flops <= 0):
        raise ValueError('positive finite compute budgets required')
    if (values == target_flops).any():
        return 'measured'
    return 'interpolated' if values.min() < target_flops < values.max() else 'extrapolated'


In [ ]:
import numpy as np
np.testing.assert_allclose(bits_per_byte([np.log(2), 3*np.log(2)], 2), 2)
assert bits_per_byte([0, 0], 3) == 0
# Same content, different number of prediction units: sum losses first.
np.testing.assert_allclose(bits_per_byte([np.log(2)]*8, 4), bits_per_byte([2*np.log(2)]*4, 4))
assert logit_storage(10, 1000) == 40000
assert logit_storage(10, 1000, 5) == 400
assert logit_storage(10, 1000, 1000) == 80000
assert logit_storage(45, 256) == 46080
_counts, _times = [10, 90], [100., 300.]
assert serving_summary(_counts, _times) == {'bytes_per_second': 250., 'p95_ms': 300.}
assert serving_summary([1]*20, list(range(1,21)))['p95_ms'] == 19
assert _counts == [10,90] and _times == [100.,300.]
for _budget, _region in [(10,'measured'),(20,'measured'),(15,'interpolated'),(9,'extrapolated'),(100,'extrapolated')]:
    assert evidence_region([20, 10, 20], _budget) == _region
assert evidence_region([10], 11) == 'extrapolated'
for _call in [
    lambda: bits_per_byte([], 1), lambda: bits_per_byte([-1], 1),
    lambda: bits_per_byte([np.nan], 1), lambda: bits_per_byte([1], 0),
    lambda: bits_per_byte([1], 1.5), lambda: bits_per_byte([1], True),
    lambda: logit_storage(2.5, 256), lambda: logit_storage(1, 0),
    lambda: logit_storage(1, 256, 257), lambda: logit_storage(1, 256, True),
    lambda: serving_summary([1], [0]), lambda: serving_summary([1.5], [1]),
    lambda: serving_summary([1], [1, 2]), lambda: serving_summary([], []),
    lambda: serving_summary([1], [np.inf]), lambda: evidence_region([], 1),
    lambda: evidence_region([0, 1], 1), lambda: evidence_region([1], np.nan),
]:
    try:
        _call()
    except ValueError:
        pass
    else:
        raise AssertionError('invalid measurement must raise ValueError')
print('BPB denominator, dense/sparse payload, sequential throughput, p95 and evidence regions passed.')


## Experiment and interpretation

Predict the result before running. Change one variable, repeat, and record what changed. Separate measured outcomes from assumptions.


In [ ]:
"""Original synthetic observations: exercise evidence accounting, not paper data."""
import json
from pathlib import Path
import runpy
import numpy as np

def run_evaluation_experiment(metrics=None):
    if metrics is None:
        file = Path(__file__).resolve().parents[1] / 'public/content/solutions/c9/10_byte_evaluation.py'
        metrics = runpy.run_path(str(file))
    budgets = np.array([10., 20., 40.])
    observed_scores = np.array([40., 50., 54.])
    # Fit y = a + b/C using ONLY the first two observations, then hold out C=40.
    design = np.column_stack([np.ones(2), 1/budgets[:2]])
    a, b = np.linalg.lstsq(design, observed_scores[:2], rcond=None)[0]
    held_out_prediction = a + b/40
    projections = []
    for noise in [-1., 0., 1.]:
        perturbed = observed_scores[:2] + np.array([0., noise])
        intercept, slope = np.linalg.lstsq(design, perturbed, rcond=None)[0]
        projections.append({'second_observation_delta': noise, 'forecast_at_100': float(intercept+slope/100),
                            'region_relative_to_fit_data': metrics['evidence_region'](budgets[:2], 100)})
    return {'source': 'synthetic teaching observations; not measurements from the paper',
            'observations': list(zip(budgets.tolist(), observed_scores.tolist())),
            'held_out_C40': {'predicted': float(held_out_prediction), 'observed': 54., 'absolute_error': float(abs(held_out_prediction-54))},
            'extrapolation_sensitivity': projections,
            'representation_BPB': metrics['bits_per_byte']([np.log(2)]*8, 4),
            'same_content_target_payload_bytes': {'dense_token': metrics['logit_storage'](10, 1000),
                                                'top5_token': metrics['logit_storage'](10, 1000, 5),
                                                'dense_byte': metrics['logit_storage'](45, 256)},
            'sequential_serving_example': metrics['serving_summary']([10, 90], [100, 300])}

if __name__ == '__main__':
    METRICS = globals() if 'bits_per_byte' in globals() else None
    RESULTS = run_evaluation_experiment(METRICS)
    print(json.dumps(RESULTS, indent=2))
    print('A two-point fit is not a validated scaling law. The omitted observation tests the fit once; changing a point exposes extrapolation sensitivity. No benchmark scores or timings above are real model measurements.')
